In [36]:
import pandas as pd

# Load the stock prices dataset
df = pd.read_csv("2) Stock Prices Data Set.csv")

# Display first 5 rows
df.head()

,symbol,date,open,high,low,close,volume
0,AAL,2014-01-02,25.0700,25.8200,25.0600,25.3600,8998943
1,AAPL,2014-01-02,79.3828,79.5756,78.8601,79.0185,58791957
2,AAP,2014-01-02,110.3600,111.8800,109.2900,109.7400,542711
3,ABBV,2014-01-02,52.1200,52.3300,51.5200,51.9800,4569061
4,ABC,2014-01-02,70.1100,70.2300,69.4800,69.8900,1148391


In [37]:
# Check number of rows and columns
df.shape

(497472, 7)

In [38]:
# Check column names, data types, and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 497472 entries, 0 to 497471
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   symbol  497472 non-null  str    
 1   date    497472 non-null  str    
 2   open    497461 non-null  float64
 3   high    497464 non-null  float64
 4   low     497464 non-null  float64
 5   close   497472 non-null  float64
 6   volume  497472 non-null  int64  
dtypes: float64(4), int64(1), str(2)
memory usage: 26.6 MB


In [39]:
# Check missing values in each column
df.isnull().sum()

symbol     0
date       0
open      11
high       8
low        8
close      0
volume     0
dtype: int64

In [40]:
# Check duplicate rows
df.duplicated().sum()

np.int64(0)

In [41]:
# Create a copy so the original dataset remains unchanged
df_clean = df.copy()

In [42]:
# Standardize column names
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df_clean.columns

Index(['symbol', 'date', 'open', 'high', 'low', 'close', 'volume'], dtype='str')

In [43]:
# Convert date column to datetime format
df_clean["date"] = pd.to_datetime(df_clean["date"], errors="coerce")

# Check data types again
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 497472 entries, 0 to 497471
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   symbol  497472 non-null  str           
 1   date    497472 non-null  datetime64[us]
 2   open    497461 non-null  float64       
 3   high    497464 non-null  float64       
 4   low     497464 non-null  float64       
 5   close   497472 non-null  float64       
 6   volume  497472 non-null  int64         
dtypes: datetime64[us](1), float64(4), int64(1), str(1)
memory usage: 26.6 MB


In [44]:
# Sort dataset by stock symbol and date
df_clean = df_clean.sort_values(by=["symbol", "date"])

df_clean.head()

,symbol,date,open,high,low,close,volume
57,A,2014-01-02,57.10,57.100,56.15,56.21,1916160
540,A,2014-01-03,56.39,57.345,56.26,56.92,1866651
1023,A,2014-01-06,57.40,57.700,56.56,56.64,1777472
1506,A,2014-01-07,56.95,57.630,56.93,57.45,1463208
1989,A,2014-01-08,57.33,58.540,57.17,58.39,2659468


In [45]:
# Display rows where open, high, or low has missing values
missing_rows = df_clean[df_clean[["open", "high", "low"]].isnull().any(axis=1)]

missing_rows

,symbol,date,open,high,low,close,volume
442107,BHF,2017-07-26,NaN,NaN,NaN,69.0842,3
188547,DHR,2015-07-17,NaN,88.76,88.24,88.7200,2056819
249223,DHR,2016-01-12,NaN,NaN,NaN,88.5500,0
188578,ES,2015-07-17,NaN,48.49,47.85,47.9200,1246786
308365,FTV,2016-07-01,NaN,NaN,NaN,49.5400,0
188760,O,2015-07-17,NaN,47.31,46.83,46.9900,1229513
249438,O,2016-01-12,NaN,NaN,NaN,52.4300,0
175557,REGN,2015-06-09,NaN,NaN,NaN,526.0900,12135
278801,UA,2016-04-07,NaN,NaN,NaN,41.5600,0
166348,VRTX,2015-05-12,NaN,NaN,NaN,124.0800,569747


In [46]:
# Fill missing open values using previous value within the same stock symbol
df_clean["open"] = df_clean.groupby("symbol")["open"].ffill()

# If any open values are still missing, use backward fill within the same stock symbol
df_clean["open"] = df_clean.groupby("symbol")["open"].bfill()

# Fill missing high values using the maximum available price in that row
df_clean["high"] = df_clean["high"].fillna(
    df_clean[["open", "close", "low"]].max(axis=1)
)

# Fill missing low values using the minimum available price in that row
df_clean["low"] = df_clean["low"].fillna(
    df_clean[["open", "close", "high"]].min(axis=1)
)

In [47]:
# Remove duplicate rows
df_clean = df_clean.drop_duplicates()

# Confirm duplicates after cleaning
df_clean.duplicated().sum()

np.int64(0)

In [48]:
# Standardize stock symbols
df_clean["symbol"] = df_clean["symbol"].str.strip().str.upper()

df_clean["symbol"].head()

57      A
540     A
1023    A
1506    A
1989    A
Name: symbol, dtype: str

In [49]:
# Check missing values after cleaning
df_clean.isnull().sum()

symbol    0
date      0
open      0
high      0
low       0
close     0
volume    0
dtype: int64

In [50]:
# Check final dataset information
df_clean.info()

<class 'pandas.DataFrame'>
Index: 497472 entries, 57 to 497471
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   symbol  497472 non-null  str           
 1   date    497472 non-null  datetime64[us]
 2   open    497472 non-null  float64       
 3   high    497472 non-null  float64       
 4   low     497472 non-null  float64       
 5   close   497472 non-null  float64       
 6   volume  497472 non-null  int64         
dtypes: datetime64[us](1), float64(4), int64(1), str(1)
memory usage: 30.4 MB


In [51]:
df_clean.to_csv("cleaned_stock_prices.csv", index=False)

### Task 1: Data Cleaning and Preprocessing Summary

The stock prices dataset was loaded using pandas. The dataset contained 497,472 rows and 7 columns. I inspected the dataset structure, column data types, missing values, and duplicate rows.

The columns `open`, `high`, and `low` contained missing values. The `open` column had 11 missing values, while the `high` and `low` columns each had 8 missing values. No duplicate rows were found.

To clean the dataset, I converted the `date` column from text into datetime format because stock price data is time-based. I sorted the data by `symbol` and `date` to make sure the stock prices followed the correct order. Missing `open` values were filled using forward fill and backward fill within each stock symbol. Missing `high` values were filled using the highest available price in the same row, while missing `low` values were filled using the lowest available price in the same row.

I also standardized the `symbol` column by removing extra spaces and converting values to uppercase. After cleaning, the dataset had no missing values and no duplicate rows. The cleaned dataset was saved as `cleaned_stock_prices.csv` for future analysis.